In [ ]:
#FIRST WE HAVE TO IMPORT ALL THE MODULE AND DEPENDENCY
from typing import TypedDict, NotRequired

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

c:\Users\Sangram Mishra\Desktop\ADK\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [3]:
#nOW WE WILL Create some dcuments using the langchain 
documents  = [
    Document(
        page_content="""
        Producer Enterprise financial assistance is provided
        for product planning, training, equipment and machinery,
        software and IT, management and marketing activities.
        The maximum admissible assistance is ₹35 lakh.
        """,
        metadata= {
            "scheme": "PE",
            "section": "Financial Assistance",
            "page": 12
        }
    ),
     Document(
        page_content="""
        Producer Enterprise training support includes
        capacity building, entrepreneurship development,
        bookkeeping, digital skills and marketing training.
        """,
        metadata={
            "scheme": "PE",
            "section": "Training",
            "page": 15
        }
    ),

    Document(
        page_content="""
        Producer Enterprise equipment and machinery support
        covers processing equipment, packaging equipment,
        storage equipment and other approved machinery.
        """,
        metadata={
            "scheme": "PE",
            "section": "Equipment",
            "page": 18
        }
    ),

    Document(
        page_content="""
        Producer Enterprise marketing support includes
        exhibitions, institutional linkages, ecommerce,
        ONDC, GeM and promotional activities.
        """,
        metadata={
            "scheme": "PE",
            "section": "Marketing",
            "page": 22
        }
    ),

    Document(
        page_content="""
        Producer Groups receive support for capacity building,
        business development and livelihood activities.
        """,
        metadata={
            "scheme": "PG",
            "section": "Financial Assistance",
            "page": 30
        }
    ),

    Document(
        page_content="""
        Retail outlets supported by ORMAS focus on product
        display, branding, billing, inventory management
        and sales monitoring.
        """,
        metadata={
            "scheme": "Retail",
            "section": "Retail Operations",
            "page": 45
        }
    )
]

In [ ]:
#these are created mannualy but actually we didn it throgh langchain document loaed and some pre defiedned dat loding pipeline
print(documents)

[Document(metadata={'scheme': 'PE', 'section': 'Financial Assistance', 'page': 12}, page_content='\n        Producer Enterprise financial assistance is provided\n        for product planning, training, equipment and machinery,\n        software and IT, management and marketing activities.\n        The maximum admissible assistance is ₹35 lakh.\n        '), Document(metadata={'scheme': 'PE', 'section': 'Training', 'page': 15}, page_content='\n        Producer Enterprise training support includes\n        capacity building, entrepreneurship development,\n        bookkeeping, digital skills and marketing training.\n        '), Document(metadata={'scheme': 'PE', 'section': 'Equipment', 'page': 18}, page_content='\n        Producer Enterprise equipment and machinery support\n        covers processing equipment, packaging equipment,\n        storage equipment and other approved machinery.\n        '), Document(metadata={'scheme': 'PE', 'section': 'Marketing', 'page': 22}, page_content='\n   

In [6]:
#now we will create the embeding model
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1384.87it/s]


In [7]:
#now we will create the chorma vector data base
vectorstore  = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="ormas_day6"
)

In [8]:
# Create vector retriever
# We retrieve more candidates than we finally want.
# For example:
# Retriever → Top 5
# Reranker → Top 3

In [10]:
vector_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k":5
    }
)

In [11]:
# Create BM25 keyword search
# This is our lexical/keyword retriever.
tokenized_documents  = [
    doc.page_content.lower().split() for doc
    in documents
]

In [14]:
#now we will pass these tokenized documents to BM25 
bm25 = BM25Okapi(tokenized_documents)

In [ ]:
def key_word_search(query,k=5):
    query_token = query.lower().split()
    scores = bm25.get_scores(query_token)

    ranked_indices = sorted(
        range(len(scores)),key= lambda i:scores[i],
        reverse=True
    )
    #print(ranked_indices)
    return [
        documents[i] for i in ranked_indices[:k]
    ]

In [31]:
# Vector search function
def vector_search(query, k=5):
    return vector_retriever.invoke(query)

In [32]:
# The text you selected explains Reciprocal Rank Fusion 
# (RRF), which is a mathematical technique used in hybrid 
# search systems to combine results from different retrieval 
# methods into a single, unified ranking.In the context of 
# the page, you have two different lists of 
# search results:Vector Search: Ranks documents based on 
# semantic meaning (understanding the intent).
# BM25 (Keyword) Search: Ranks documents based on 
# exact word matches.Because these two methods score and 
# rank documents entirely differently, you cannot just add 
# their raw scores together. RRF solves this by scoring 
# documents based strictly on their position (rank) in 
# each list.The Formula Breakdown
# :$$RRF(d) = \sum_{r} \frac{1}{k + rank_r(d)}$$$d$: 
# The specific document you are scoring.$r$: 
# The different search result lists (in your case, 
# the Vector list and the BM25 list).$rank_r(d)$: 
# The position of document $d$ in list 
# $r$ (e.g., 1st, 2nd, 3rd).$k$: A constant value 
# (usually set to 60). This acts as a smoothing factor
# so that a document ranking #1 in 
# one list doesn't completely overpower a document 
# that consistently ranks #2 or #3 across multiple lists.
# How it works in practice:If your "Financial Assistance" 
# document ranks #1 in the Vector Search and 
# #3 in the BM25 Search, its final RRF score is calculated 
# by adding those inverse ranks together 
# (assuming $k = 60$):$$Score = \frac{1}{60 + 1} + \frac{1}{60 + 3} = \frac{1}{61} + \frac{1}{63}
# $$The system calculates this score for every document that appears in your searches, adds up the fractions, 
# and then sorts the documents from highest to lowest score.
#  This ensures the final list highlights documents that perform well across multiple search strategies.

In [35]:
def reciprocal_rank_fusion(
        result_lists,
        k = 60
):
    scores = {}
    doc_lookup = {}

    for result in result_lists:
        for rank ,doc in enumerate(result,start=1):
            print(f"doc {doc} rank is {rank}")
            doc_id = (doc.metadata.get("source"),
                      doc.metadata.get("page"),
                      doc.page_content)
            doc_lookup[doc_id] = doc
            print(f"doc_lookup is {doc_lookup}")

            if doc_id not in scores:
                scores[doc_id] = 0

            scores[doc_id] += (
                1/(k+rank)
            )
            print(f"scores is {scores}")
    ranked_docs = sorted(
            scores.items(),
            key = lambda x:x[1],
            reverse= True
        )
    return [
        doc_lookup[doc_id] for 
        doc_id, scores in ranked_docs
    ]



In [36]:
# Test hybrid retrieval
query = "What financial assistance is available for PE?"

vector_results = vector_search(
    query,
    k=5
)

keyword_results = key_word_search(
    query,
    k=5
)

hybrid_results = reciprocal_rank_fusion(
    [
        vector_results,
        keyword_results
    ]
)

[0, 4, 1, 2, 3, 5]
doc page_content='
        Producer Enterprise financial assistance is provided
        for product planning, training, equipment and machinery,
        software and IT, management and marketing activities.
        The maximum admissible assistance is ₹35 lakh.
        ' metadata={'section': 'Financial Assistance', 'page': 12, 'scheme': 'PE'} rank is 1
doc_lookup is {(None, 12, '\n        Producer Enterprise financial assistance is provided\n        for product planning, training, equipment and machinery,\n        software and IT, management and marketing activities.\n        The maximum admissible assistance is ₹35 lakh.\n        '): Document(id='3a6dde8d-cd46-4c6a-9aba-6a150c3b6217', metadata={'section': 'Financial Assistance', 'page': 12, 'scheme': 'PE'}, page_content='\n        Producer Enterprise financial assistance is provided\n        for product planning, training, equipment and machinery,\n        software and IT, management and marketing activities.\n     

In [37]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

c:\Users\Sangram Mishra\Desktop\ADK\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sangram Mishra\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5523.94it/s]


In [39]:
def rerank_documents(
    query,
    documents,
    top_k=3
):

    pairs = [
        (
            query,
            doc.page_content
        )
        for doc in documents
    ]

    scores = reranker.predict(
        pairs
    )

    scored_documents = list(
        zip(documents, scores)
    )

    scored_documents.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return [
        doc
        for doc, score
        in scored_documents[:top_k]
    ]

In [40]:
reranked_results = rerank_documents(
    query,
    hybrid_results,
    top_k=3
)

In [42]:
def build_context(documents):

    context_parts = []

    for doc in documents:

        section = doc.metadata.get(
            "section",
            "Unknown"
        )

        page = doc.metadata.get(
            "page",
            "Unknown"
        )

        context_parts.append(
            f"""
Source Section: {section}
Page: {page}

{doc.page_content}
"""
        )

    return "\n\n".join(
        context_parts
    )

In [45]:
prompt = ChatPromptTemplate.from_template(
    """
You are an ORMAS knowledge assistant.

Answer the question using ONLY the
provided context.

If the answer is not present in the
context, say:

"I could not find this information
in the provided documents."

Do not invent information.

Mention the source section and page
when possible.

-------------------------
CONTEXT
-------------------------

{context}

-------------------------
QUESTION
-------------------------

{question}

-------------------------
ANSWER
-------------------------
"""
)

In [43]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [44]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(model="openai/gpt-oss-20b",
                        model_provider="groq")

In [46]:
def generate_answer(
    query,
    documents
):

    context = build_context(
        documents
    )

    messages = prompt.invoke(
        {
            "context": context,
            "question": query
        }
    )

    response = llm.invoke(
        messages
    )

    return response.content

In [47]:
def day6_rag(query):

    # ==========================================
    # STEP 1 — VECTOR SEARCH
    # ==========================================

    vector_results = vector_search(
        query,
        k=5
    )


    # ==========================================
    # STEP 2 — KEYWORD SEARCH
    # ==========================================

    keyword_results = key_word_search(
        query,
        k=5
    )


    # ==========================================
    # STEP 3 — HYBRID SEARCH / RRF
    # ==========================================

    hybrid_results = reciprocal_rank_fusion(
        [
            vector_results,
            keyword_results
        ]
    )


    # ==========================================
    # STEP 4 — RERANKING
    # ==========================================

    reranked_results = rerank_documents(
        query,
        hybrid_results,
        top_k=3
    )


    # ==========================================
    # STEP 5 — BUILD CONTEXT
    # ==========================================

    context = build_context(
        reranked_results
    )


    # ==========================================
    # STEP 6 — PROMPT
    # ==========================================

    messages = prompt.invoke(
        {
            "context": context,
            "question": query
        }
    )


    # ==========================================
    # STEP 7 — LLM
    # ==========================================

    response = llm.invoke(
        messages
    )


    # ==========================================
    # STEP 8 — RETURN EVERYTHING
    # ==========================================

    return {
        "answer": response.content,
        "documents": reranked_results,
        "context": context
    }

In [48]:
query = """
What is the maximum financial assistance
available for Producer Enterprise?
"""

result = day6_rag(query)

print("\nANSWER:")
print(result["answer"])

[0, 4, 3, 1, 2, 5]
doc page_content='
        Producer Enterprise financial assistance is provided
        for product planning, training, equipment and machinery,
        software and IT, management and marketing activities.
        The maximum admissible assistance is ₹35 lakh.
        ' metadata={'section': 'Financial Assistance', 'scheme': 'PE', 'page': 12} rank is 1
doc_lookup is {(None, 12, '\n        Producer Enterprise financial assistance is provided\n        for product planning, training, equipment and machinery,\n        software and IT, management and marketing activities.\n        The maximum admissible assistance is ₹35 lakh.\n        '): Document(id='3a6dde8d-cd46-4c6a-9aba-6a150c3b6217', metadata={'section': 'Financial Assistance', 'scheme': 'PE', 'page': 12}, page_content='\n        Producer Enterprise financial assistance is provided\n        for product planning, training, equipment and machinery,\n        software and IT, management and marketing activities.\n     

In [49]:
query = """
What is the maximum financial assistance
available for Producer Enterprise?
"""

result = day6_rag(query)

print("\nANSWER:")
print(result["answer"])

[0, 4, 3, 1, 2, 5]
doc page_content='
        Producer Enterprise financial assistance is provided
        for product planning, training, equipment and machinery,
        software and IT, management and marketing activities.
        The maximum admissible assistance is ₹35 lakh.
        ' metadata={'section': 'Financial Assistance', 'page': 12, 'scheme': 'PE'} rank is 1
doc_lookup is {(None, 12, '\n        Producer Enterprise financial assistance is provided\n        for product planning, training, equipment and machinery,\n        software and IT, management and marketing activities.\n        The maximum admissible assistance is ₹35 lakh.\n        '): Document(id='3a6dde8d-cd46-4c6a-9aba-6a150c3b6217', metadata={'section': 'Financial Assistance', 'page': 12, 'scheme': 'PE'}, page_content='\n        Producer Enterprise financial assistance is provided\n        for product planning, training, equipment and machinery,\n        software and IT, management and marketing activities.\n     